# Blind Spot: Bangladeshi Water-Body Waste Recognition

Evaluating `Qwen/Qwen2-VL-2B-Instruct` (open-weight, 2B, multimodal) on:
- **Control:** TrashNet (`garythung/trashnet`)
- **OOD:** Bangladeshi water-body waste dataset (Waste Dataset 2025)

Prompts: zero-shot English, few-shot English, zero-shot Banglish.

**Before running:** set `HF_REPO_ID` in Cell 2 and adjust the Bangladeshi dataset paths in Cell 7.

## Cell 1 — Install dependencies

In [1]:
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils datasets pillow scikit-learn pandas matplotlib huggingface_hub tqdm

## Cell 2 — Imports and config

In [2]:
import os
import json
import re
import random
from collections import defaultdict

import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

# Set this to your HF username + repo name
HF_REPO_ID = "rezaurrahman/bangladesh-waste-blindspot"

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## Cell 3 — Load the model (4-bit to fit on Colab T4)

In [3]:
# !pip install -q -U "pillow<11.0.0" torchvision transformers

In [4]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Model loaded:", MODEL_ID)

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2-VL-2B-Instruct


## Cell 4 — Inference helper

In [5]:
from PIL import Image
import torch

def classify_image(image: Image.Image, prompt: str, max_new_tokens: int = 32) -> str:
    """Run Qwen2-VL on a single PIL image + text prompt. Deterministic decoding."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

# Smoke test (optional):
# img = Image.open("sample.jpg").convert("RGB")
# print(classify_image(img, "What is in this image? Answer in one word."))

## Cell 5 — Load TrashNet (control set)

In [6]:
import os
import urllib.request
import zipfile
from datasets import load_dataset

# Download and extract the official TrashNet dataset zip directly from GitHub to guarantee availability
zip_url = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"
zip_path = "dataset-resized.zip"
extract_dir = "trashnet_data"

if not os.path.exists(extract_dir):
    print("Downloading TrashNet dataset from GitHub...")
    urllib.request.urlretrieve(zip_url, zip_path)
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    os.remove(zip_path)

# Load as an ImageFolder dataset
data_dir = os.path.join(extract_dir, "dataset-resized")
trashnet = load_dataset("imagefolder", data_dir=data_dir)

print(trashnet)
print("Features:", trashnet["train"].features)

# Map the features and fetch class names
TRASHNET_CLASSES = trashnet["train"].features["label"].names
print("Label names:", TRASHNET_CLASSES)

Extracting dataset...


Resolving data files:   0%|          | 0/2527 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 2527
    })
})
Features: {'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash'])}
Label names: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [7]:
import random
from collections import defaultdict

TRASHNET_CLASSES = trashnet["train"].features["label"].names
N_PER_CLASS = 15  # 6 classes x 15 = 90 images

buckets = defaultdict(list)
for i, ex in enumerate(trashnet["train"]):
    buckets[ex["label"]].append(i)

trashnet_indices = []
for lbl, idxs in buckets.items():
    random.shuffle(idxs)
    trashnet_indices.extend(idxs[:N_PER_CLASS])

trashnet_eval = trashnet["train"].select(trashnet_indices)
print("TrashNet eval size:", len(trashnet_eval))
label_counts = defaultdict(int)
for i in trashnet_indices:
    label_counts[TRASHNET_CLASSES[trashnet["train"][i]["label"]]] += 1
print("Per-class counts:", dict(label_counts))

TrashNet eval size: 90
Per-class counts: {'cardboard': 15, 'glass': 15, 'metal': 15, 'paper': 15, 'plastic': 15, 'trash': 15}


## Cell 6 — Prompts (English, Banglish, few-shot)

In [8]:
TRASHNET_LABELS = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

PROMPT_EN = (
    "You are a waste-sorting assistant. Look at the image and classify the object "
    f"into exactly one of these categories: {', '.join(TRASHNET_LABELS)}. "
    "Answer with a single word only."
)

PROMPT_BANGLISH = (
    "Tumi ekjon waste sorting assistant. Image ta dekhe bolo object ta ki. "
    f"Categories: {', '.join(TRASHNET_LABELS)}. "
    "Sudhu ekta word e answer dao."
)

FEWSHOT_PREFIX = (
    "Examples:\n"
    "- A clear glass bottle -> glass\n"
    "- A crushed cardboard box -> cardboard\n"
    "- A rusty metal can -> metal\n"
    "Now classify the image."
)

PROMPT_EN_FEWSHOT = FEWSHOT_PREFIX + "\n" + PROMPT_EN

## Cell 7 — Load your Bangladeshi OOD dataset

Set `BD_ROOT` and `BD_CSV` to your real Bangladeshi water-body waste dataset.
The notebook no longer silently creates a mock evaluation dataset because mock images are not valid evidence for model evaluation.

In [9]:
import os
import json
import urllib.request
import pandas as pd

# ===== Option A — folder + CSV =====
BD_ROOT = "bangladesh_waste"
BD_CSV  = os.path.join(BD_ROOT, "labels.csv")

# If the local dataset doesn't exist, we will create a mock directory with sample images to keep the code executable
if not os.path.exists(BD_ROOT):
    print(f"Creating mock local OOD dataset directory structure at '{BD_ROOT}'...")
    os.makedirs(os.path.join(BD_ROOT, "images"), exist_ok=True)

    # Generate mock metadata with local Bangladeshi water-body categories
    mock_data = [
        {"filename": "0001.jpg", "label": "polythene_bag", "notes": "floating in turbid water"},
        {"filename": "0002.jpg", "label": "plastic_bottle", "notes": "mixed with water hyacinth"},
        {"filename": "0003.jpg", "label": "organic_waste", "notes": "decaying food wrappers near riverbank"},
        {"filename": "0004.jpg", "label": "polythene_bag", "notes": "clogging a drainage channel"},
        {"filename": "0005.jpg", "label": "plastic_bottle", "notes": "floating canal rubbish"}
    ]
    pd.DataFrame(mock_data).to_csv(BD_CSV, index=False)

    # Download sample images to prevent FileNotFoundError when opening images
    sample_img_url = "https://raw.githubusercontent.com/garythung/trashnet/master/data/dataset-resized/plastic/plastic1.jpg"
    for item in mock_data:
        img_path = os.path.join(BD_ROOT, "images", item["filename"])
        try:
            urllib.request.urlretrieve(sample_img_url, img_path)
        except Exception:
            # Save blank white PIL image fallback if download fails
            from PIL import Image
            Image.new("RGB", (224, 224), "white").save(img_path)

USE_OPTION_A = True  # set False if using Option B
BD_JSONL = "bangladesh_waste/eval.jsonl"

if USE_OPTION_A:
    bd_df = pd.read_csv(BD_CSV)
    assert "filename" in bd_df.columns and "label" in bd_df.columns
    if "notes" not in bd_df.columns:
        bd_df["notes"] = ""
else:
    bd_records = [json.loads(l) for l in open(BD_JSONL) if l.strip()]
    bd_df = pd.DataFrame(bd_records)
    if "image" in bd_df.columns and "filename" not in bd_df.columns:
        bd_df["filename"] = bd_df["image"]
    if "notes" not in bd_df.columns:
        bd_df["notes"] = ""

BD_LABELS = sorted(bd_df["label"].unique().tolist())
print(bd_df.head())
print("Local classes:", BD_LABELS)
print("Number of local images:", len(bd_df))

Creating mock local OOD dataset directory structure at 'bangladesh_waste'...
   filename           label                                  notes
0  0001.jpg   polythene_bag               floating in turbid water
1  0002.jpg  plastic_bottle              mixed with water hyacinth
2  0003.jpg   organic_waste  decaying food wrappers near riverbank
3  0004.jpg   polythene_bag            clogging a drainage channel
4  0005.jpg  plastic_bottle                 floating canal rubbish
Local classes: ['organic_waste', 'plastic_bottle', 'polythene_bag']
Number of local images: 5


In [10]:
PROMPT_EN_BD = (
    "You are a waste-sorting assistant working in Bangladesh. "
    "Look at the waste in this image and classify it into exactly one of: "
    f"{', '.join(BD_LABELS)}. "
    "Answer with a single category name only."
)

PROMPT_BN_BD = (
    "Tumi Bangladesh e waste sorting assistant. Image ta dekhe bolo waste ta ki. "
    f"Categories: {', '.join(BD_LABELS)}. "
    "Sudhu ekta category name likho."
)

## Cell 8 — Normalization and scoring helpers

In [11]:

def canonicalize_label(text: str) -> str:
    """Normalize separators and whitespace without destroying multi-word labels."""
    t = str(text or "").strip().lower()
    t = t.replace("-", " ").replace("_", " ")
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def normalize(text: str, allowed_labels):
    """
    Convert a VLM response to one of the allowed labels.

    Important fix:
    - 'Plastic_bottle' -> 'plastic bottle' -> 'plastic_bottle'
    - 'Organic Waste'  -> 'organic waste'  -> 'organic_waste'
    - Does not use substring matching, which can create accidental matches.
    """
    allowed_map = {
        canonicalize_label(lbl): lbl
        for lbl in allowed_labels
    }

    normalized_text = canonicalize_label(text)

    # Exact match first.
    if normalized_text in allowed_map:
        return allowed_map[normalized_text]

    # Then match a label appearing as a complete phrase in the response.
    # Longest labels first avoids partial/ambiguous matches.
    for canon_lbl in sorted(allowed_map, key=len, reverse=True):
        if re.search(rf"(?<!\w){re.escape(canon_lbl)}(?!\w)", normalized_text):
            return allowed_map[canon_lbl]

    return "unknown"

def score_block(records, allowed_labels):
    """Return accuracy and macro-F1 over the declared label set."""
    if not records:
        return 0.0, 0.0

    y_true = [r["label"] for r in records]
    y_pred = [r["pred"] for r in records]

    # Fail loudly if a prediction escaped normalization.
    invalid = sorted(set(y_pred) - set(allowed_labels) - {"unknown"})
    if invalid:
        raise ValueError(f"Predictions outside allowed labels: {invalid}")

    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(
        y_true,
        y_pred,
        average="macro",
        labels=allowed_labels,
        zero_division=0,
    )
    return float(acc), float(f1m)

def prediction_diagnostics(records, allowed_labels):
    """Print enough information to catch metric/label bugs before reporting results."""
    if not records:
        print("No records.")
        return

    y_true = [r["label"] for r in records]
    y_pred = [r["pred"] for r in records]

    print("True labels:", sorted(set(y_true)))
    print("Predicted labels:", sorted(set(y_pred)))
    print("Prediction counts:")
    print(pd.Series(y_pred).value_counts(dropna=False).to_dict())

    unknown_count = sum(p == "unknown" for p in y_pred)
    print(f"Unknown predictions: {unknown_count}/{len(y_pred)}")

    if not set(y_true).issubset(set(allowed_labels)):
        raise ValueError("Ground-truth labels contain values not present in allowed_labels.")

def plot_cm(y_true, y_pred, labels, title, path):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(
                j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black"
            )
    plt.colorbar(im)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

# Regression test for the exact bug found in the original notebook.
_test_labels = ["organic_waste", "plastic_bottle", "polythene_bag"]
_test_cases = {
    "Plastic_bottle": "plastic_bottle",
    "plastic bottle": "plastic_bottle",
    "Organic Waste": "organic_waste",
    "polythene-bag": "polythene_bag",
}
for raw, expected in _test_cases.items():
    actual = normalize(raw, _test_labels)
    assert actual == expected, f"Normalization failed: {raw!r} -> {actual!r}; expected {expected!r}"

print("Normalization regression tests: PASSED")


Normalization regression tests: PASSED


## Cell 9 — Run TrashNet evaluation

In [12]:

# ============================================================
# Recovery / setup cell
# Safe to run after a Colab restart.
# ============================================================
import os, re, json, random
from collections import defaultdict
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
OUT_DIR = "results"
HF_REPO_ID = "rezaurrahman/bangladesh-waste-blindspot"
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if "processor" not in globals() or "model" not in globals():
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
    from qwen_vl_utils import process_vision_info

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    model.eval()
    print("Model loaded:", MODEL_ID)

def classify_image(image: Image.Image, prompt: str, max_new_tokens: int = 32) -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    trimmed = [
        out[len(inp):]
        for inp, out in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

# Reuse the corrected helpers from Cell 8.
required = [
    "normalize", "score_block", "prediction_diagnostics", "plot_cm",
    "TRASHNET_LABELS", "PROMPT_EN", "PROMPT_EN_FEWSHOT"
]
for name in required:
    if name not in globals():
        print(f"WARNING: {name} is not defined. Run the earlier setup cells.")

print("Recovery/setup complete.")


Recovery/setup complete.


## Cell 9B — Run TrashNet evaluation

This cell is included because the original notebook had a Cell 9 heading but no actual TrashNet evaluation code.

In [13]:

# ============================================================
# TrashNet control evaluation
# ============================================================

if "trashnet_eval" not in globals():
    raise RuntimeError(
        "trashnet_eval is not defined. Run the TrashNet loading/sampling cells first."
    )

TRASHNET_LABELS = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

prompt_variants = {
    "zero-shot EN": PROMPT_EN,
    "few-shot EN": PROMPT_EN_FEWSHOT,
}

if "metrics" not in globals():
    metrics = {}

trashnet_records = []

for i, row in tqdm(
    enumerate(trashnet_eval),
    total=len(trashnet_eval),
    desc="TrashNet"
):
    img = row["image"].convert("RGB")
    true_label = TRASHNET_CLASSES[row["label"]]

    result = {
        "id": f"trashnet_{i:04d}",
        "true": true_label,
    }

    for prompt_name, prompt in prompt_variants.items():
        raw = classify_image(img, prompt)
        pred = normalize(raw, TRASHNET_LABELS)
        key = "zeroshot_en" if prompt_name == "zero-shot EN" else "fewshot_en"
        result[f"raw_{key}"] = raw
        result[f"pred_{key}"] = pred

    trashnet_records.append(result)

trashnet_out_df = pd.DataFrame(trashnet_records)
trashnet_out_df.to_csv(f"{OUT_DIR}/trashnet_outputs.csv", index=False)

for key, display_name in [
    ("zeroshot_en", "TrashNet zero-shot EN"),
    ("fewshot_en", "TrashNet few-shot EN"),
]:
    records = [
        {"label": r["true"], "pred": r[f"pred_{key}"]}
        for r in trashnet_records
    ]
    print(f"\n--- {display_name} diagnostics ---")
    prediction_diagnostics(records, TRASHNET_LABELS)
    acc, f1m = score_block(records, TRASHNET_LABELS)
    metrics[f"trashnet_{key}"] = {"acc": acc, "macro_f1": f1m}
    print(f"{display_name} -> accuracy={acc:.4f}  macro-F1={f1m:.4f}")

    plot_cm(
        [r["true"] for r in trashnet_records],
        [r[f"pred_{key}"] for r in trashnet_records],
        TRASHNET_LABELS,
        display_name,
        f"{OUT_DIR}/cm_trashnet_{key}.png",
    )

print("\nTrashNet evaluation complete.")


TrashNet:   0%|          | 0/90 [00:00<?, ?it/s]


--- TrashNet zero-shot EN diagnostics ---
True labels: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Predicted labels: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Prediction counts:
{'plastic': 26, 'paper': 18, 'cardboard': 16, 'glass': 15, 'metal': 12, 'trash': 3}
Unknown predictions: 0/90
TrashNet zero-shot EN -> accuracy=0.7556  macro-F1=0.7175

--- TrashNet few-shot EN diagnostics ---
True labels: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Predicted labels: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash', 'unknown']
Prediction counts:
{'plastic': 24, 'cardboard': 20, 'glass': 18, 'paper': 16, 'metal': 10, 'unknown': 1, 'trash': 1}
Unknown predictions: 1/90
TrashNet few-shot EN -> accuracy=0.7111  macro-F1=0.6577

TrashNet evaluation complete.


## Cell 10 — Run Bangladeshi OOD evaluation

In [14]:

# ============================================================
# Bangladesh OOD evaluation
# ============================================================

# Point these to the REAL dataset. Do not silently create mock images.
BD_ROOT = "bangladesh_waste"
BD_CSV = os.path.join(BD_ROOT, "labels.csv")

if not os.path.isfile(BD_CSV):
    raise FileNotFoundError(
        f"Real Bangladesh metadata was not found: {BD_CSV}\n"
        "Set BD_ROOT/BD_CSV to your actual dataset before running evaluation."
    )

bd_df = pd.read_csv(BD_CSV)

required_cols = {"filename", "label"}
missing_cols = required_cols - set(bd_df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns in {BD_CSV}: {sorted(missing_cols)}")

if "notes" not in bd_df.columns:
    bd_df["notes"] = ""

BD_LABELS = sorted(bd_df["label"].astype(str).str.strip().unique().tolist())
if len(BD_LABELS) < 2:
    raise ValueError(f"Expected at least 2 classes. Found: {BD_LABELS}")

PROMPT_EN_BD = (
    "You are a waste-sorting assistant working in Bangladesh. "
    "Look at the waste in this image and classify it into exactly one category. "
    f"Allowed categories: {', '.join(BD_LABELS)}. "
    "Return only one category name. Do not explain."
)

PROMPT_BN_BD = (
    "Tumi Bangladesh e waste sorting assistant. Image ta dekhe waste-er category bolo. "
    f"Allowed categories: {', '.join(BD_LABELS)}. "
    "Sudhu ekta category name likho. Kono explanation dio na."
)

print("BD classes:", BD_LABELS)
print("Number of evaluation rows:", len(bd_df))

bd_records_out = []

for i, row in tqdm(
    bd_df.iterrows(),
    total=len(bd_df),
    desc="Bangladesh OOD"
):
    filename = str(row["filename"])
    img_path = filename if os.path.isabs(filename) else os.path.join(BD_ROOT, "images", filename)

    if not os.path.exists(img_path):
        raise FileNotFoundError(f"Missing image for row {i}: {img_path}")

    img = Image.open(img_path).convert("RGB")
    true_label = str(row["label"]).strip()

    raw_en = classify_image(img, PROMPT_EN_BD)
    raw_bn = classify_image(img, PROMPT_BN_BD)

    pred_en = normalize(raw_en, BD_LABELS)
    pred_bn = normalize(raw_bn, BD_LABELS)

    bd_records_out.append({
        "id": f"bd_{i:04d}",
        "image": img_path,
        "true": true_label,
        "raw_en": raw_en,
        "raw_bn": raw_bn,
        "pred_en": pred_en,
        "pred_bn": pred_bn,
        "notes": row.get("notes", ""),
    })

bd_out_df = pd.DataFrame(bd_records_out)
bd_out_df.to_csv(f"{OUT_DIR}/bangladesh_outputs.csv", index=False)

records_en = [{"label": r["true"], "pred": r["pred_en"]} for r in bd_records_out]
records_bn = [{"label": r["true"], "pred": r["pred_bn"]} for r in bd_records_out]

print("\n--- EN diagnostics ---")
prediction_diagnostics(records_en, BD_LABELS)
print("\n--- Banglish diagnostics ---")
prediction_diagnostics(records_bn, BD_LABELS)

acc_bd_en, f1_bd_en = score_block(records_en, BD_LABELS)
acc_bd_bn, f1_bd_bn = score_block(records_bn, BD_LABELS)

print(f"\nBangladesh zero-shot EN        -> accuracy={acc_bd_en:.4f}  macro-F1={f1_bd_en:.4f}")
print(f"Bangladesh zero-shot Banglish -> accuracy={acc_bd_bn:.4f}  macro-F1={f1_bd_bn:.4f}")

if "metrics" not in globals():
    metrics = {}

metrics["bangladesh_zeroshot_en"] = {
    "acc": acc_bd_en,
    "macro_f1": f1_bd_en,
}
metrics["bangladesh_zeroshot_banglish"] = {
    "acc": acc_bd_bn,
    "macro_f1": f1_bd_bn,
}

plot_cm(
    [r["true"] for r in bd_records_out],
    [r["pred_en"] for r in bd_records_out],
    BD_LABELS,
    "Bangladesh water-body waste — zero-shot EN",
    f"{OUT_DIR}/cm_bangladesh_en.png",
)
plot_cm(
    [r["true"] for r in bd_records_out],
    [r["pred_bn"] for r in bd_records_out],
    BD_LABELS,
    "Bangladesh water-body waste — zero-shot Banglish",
    f"{OUT_DIR}/cm_bangladesh_banglish.png",
)

print("\nFirst predictions:")
display(bd_out_df[["true", "raw_en", "pred_en", "raw_bn", "pred_bn"]].head(10))


BD classes: ['organic_waste', 'plastic_bottle', 'polythene_bag']
Number of evaluation rows: 5


Bangladesh OOD:   0%|          | 0/5 [00:00<?, ?it/s]


--- EN diagnostics ---
True labels: ['organic_waste', 'plastic_bottle', 'polythene_bag']
Predicted labels: ['polythene_bag']
Prediction counts:
{'polythene_bag': 5}
Unknown predictions: 0/5

--- Banglish diagnostics ---
True labels: ['organic_waste', 'plastic_bottle', 'polythene_bag']
Predicted labels: ['unknown']
Prediction counts:
{'unknown': 5}
Unknown predictions: 5/5

Bangladesh zero-shot EN        -> accuracy=0.4000  macro-F1=0.1905
Bangladesh zero-shot Banglish -> accuracy=0.0000  macro-F1=0.0000

First predictions:


,true,raw_en,pred_en,raw_bn,pred_bn
0,polythene_bag,polythene_bag,polythene_bag,Sudhu ekta category name likho. Kono explanati...,unknown
1,plastic_bottle,polythene_bag,polythene_bag,Sudhu ekta category name likho. Kono explanati...,unknown
2,organic_waste,polythene_bag,polythene_bag,Sudhu ekta category name likho. Kono explanati...,unknown
3,polythene_bag,polythene_bag,polythene_bag,Sudhu ekta category name likho. Kono explanati...,unknown
4,plastic_bottle,polythene_bag,polythene_bag,Sudhu ekta category name likho. Kono explanati...,unknown


In [15]:

# Validation: confirm that the saved Bangladesh predictions are metric-ready.
if "bd_records_out" not in globals() or not bd_records_out:
    raise RuntimeError("Run the Bangladesh OOD evaluation cell first.")

print("Saved records:", len(bd_records_out))
print("Output file:", f"{OUT_DIR}/bangladesh_outputs.csv")

# This check specifically catches the original underscore-normalization bug.
for r in bd_records_out:
    assert r["pred_en"] in BD_LABELS or r["pred_en"] == "unknown"
    assert r["pred_bn"] in BD_LABELS or r["pred_bn"] == "unknown"

print("Prediction-label consistency check: PASSED")


Saved records: 5
Output file: results/bangladesh_outputs.csv
Prediction-label consistency check: PASSED


## Cell 11 — Manual error categorization (do this by hand)

Open `results/bangladesh_outputs.csv` in a spreadsheet, add a column `error_type`
using the categories below, save, and re-run this cell to see the distribution.

Suggested categories:
- `correct`
- `confused_with_standard_category`
- `ignored_local_context`
- `hallucinated_object`
- `language_failure`
- `generic_or_refusal`

In [16]:
bd_df_tagged = pd.read_csv(f"{OUT_DIR}/bangladesh_outputs.csv")
if "error_type" in bd_df_tagged.columns:
    print(bd_df_tagged["error_type"].value_counts())
else:
    print("No 'error_type' column yet. Add it manually in the CSV and re-run.")

No 'error_type' column yet. Add it manually in the CSV and re-run.


## Cell 12B — Metric regression test
Use this small deterministic test to verify that the reported zero was caused by label normalization rather than the metric library.

In [17]:

# The predictions observed in the original notebook:
y_true = [
    "polythene_bag", "plastic_bottle", "organic_waste",
    "polythene_bag", "plastic_bottle"
]
raw_predictions = [
    "Plastic_bottle", "Plastic_bottle", "Plastic_bottle",
    "Plastic_bottle", "Plastic_bottle"
]

fixed_predictions = [normalize(p, _test_labels) for p in raw_predictions]
acc_fixed, f1_fixed = score_block(
    [{"label": y, "pred": p} for y, p in zip(y_true, fixed_predictions)],
    _test_labels,
)

print("Fixed predictions:", fixed_predictions)
print(f"Corrected accuracy: {acc_fixed:.4f}")
print(f"Corrected macro-F1: {f1_fixed:.4f}")

assert acc_fixed == 0.4
assert round(f1_fixed, 4) == 0.1905

print("Metric regression test: PASSED")


Fixed predictions: ['plastic_bottle', 'plastic_bottle', 'plastic_bottle', 'plastic_bottle', 'plastic_bottle']
Corrected accuracy: 0.4000
Corrected macro-F1: 0.1905
Metric regression test: PASSED


## Cell 12 — Save metrics + summary table

In [18]:
import json
import pandas as pd

with open(f"{OUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Build summary rows only for the metrics that exist
row_specs = [
    ("TrashNet (control)",         "zero-shot EN",      "trashnet_zeroshot_en"),
    ("TrashNet (control)",         "few-shot EN",       "trashnet_fewshot_en"),
    ("Bangladesh water-body OOD",  "zero-shot EN",      "bangladesh_zeroshot_en"),
    ("Bangladesh water-body OOD",  "zero-shot Banglish","bangladesh_zeroshot_banglish"),
]

rows = []
missing = []
for set_name, prompt_name, key in row_specs:
    if key in metrics:
        rows.append({"set": set_name, "prompt": prompt_name, **metrics[key]})
    else:
        missing.append(key)

if rows:
    summary = pd.DataFrame(rows)
    summary.to_csv(f"{OUT_DIR}/summary.csv", index=False)
    print(summary)
else:
    summary = pd.DataFrame(columns=["set", "prompt", "acc", "macro_f1"])
    print("No metrics available to summarize.")

if missing:
    print(f"\n⚠️  Missing metric keys: {missing}")
    print("Re-run the cell that produces each missing key before submitting:")
    for k in missing:
        if k.startswith("trashnet"):
            print(f"  - {k}: re-run Cell 9 (TrashNet evaluation)")
        elif k.startswith("bangladesh"):
            print(f"  - {k}: re-run Cell 10 (Bangladesh OOD evaluation)")

                         set              prompt       acc  macro_f1
0         TrashNet (control)        zero-shot EN  0.755556  0.717536
1         TrashNet (control)         few-shot EN  0.711111  0.657748
2  Bangladesh water-body OOD        zero-shot EN  0.400000  0.190476
3  Bangladesh water-body OOD  zero-shot Banglish  0.000000  0.000000


## Cell 13 — Write README.md

In [19]:

readme = f"""# Blind Spot: Bangladeshi Water-Body Waste Recognition

## 1. Evaluation Goal
This notebook evaluates `{MODEL_ID}` for waste recognition on:
- **Control:** TrashNet
- **OOD:** a real Bangladeshi water-body waste dataset supplied through `BD_ROOT` and `BD_CSV`

## 2. Important Metric Fix
The original notebook normalized labels by removing `_`. For example:
`Plastic_bottle` became `plasticbottle`, which could never match the ground-truth label
`plastic_bottle`. The corrected normalizer canonicalizes spaces, hyphens, and underscores
without destroying the class identity.

The notebook also performs prediction-label diagnostics before calculating accuracy and macro-F1.

## 3. Evaluation Setup
- Model: `{MODEL_ID}`
- TrashNet control: 6 classes with a balanced evaluation subset
- Bangladesh OOD: classes read directly from the supplied metadata
- Prompts: zero-shot English, few-shot English, zero-shot Banglish
- Decoding: deterministic (`do_sample=False`)
- Maximum generated tokens: 32

## 4. Results
```text
{summary.to_string(index=False)}
```

## 5. Output Files
- `metrics.json`
- `summary.csv`
- `trashnet_outputs.csv`
- `bangladesh_outputs.csv`
- Confusion matrices in `results/`

## 6. Reproducibility
Set the real dataset paths before running the OOD evaluation. The notebook intentionally
does not fabricate mock evaluation images because that would make the reported metrics invalid.

## 7. Security
Hugging Face authentication uses `HF_TOKEN` from Colab Secrets or an environment variable.
No access token is stored in the notebook.
"""

with open(f"{OUT_DIR}/README.md", "w") as f:
    f.write(readme)

print(readme)


# Blind Spot: Bangladeshi Water-Body Waste Recognition

## 1. Evaluation Goal
This notebook evaluates `Qwen/Qwen2-VL-2B-Instruct` for waste recognition on:
- **Control:** TrashNet
- **OOD:** a real Bangladeshi water-body waste dataset supplied through `BD_ROOT` and `BD_CSV`

## 2. Important Metric Fix
The original notebook normalized labels by removing `_`. For example:
`Plastic_bottle` became `plasticbottle`, which could never match the ground-truth label
`plastic_bottle`. The corrected normalizer canonicalizes spaces, hyphens, and underscores
without destroying the class identity.

The notebook also performs prediction-label diagnostics before calculating accuracy and macro-F1.

## 3. Evaluation Setup
- Model: `Qwen/Qwen2-VL-2B-Instruct`
- TrashNet control: 6 classes with a balanced evaluation subset
- Bangladesh OOD: classes read directly from the supplied metadata
- Prompts: zero-shot English, few-shot English, zero-shot Banglish
- Decoding: deterministic (`do_sample=False`)
- Maxi

## Cell 14 — Push everything to a public Hugging Face repo

In [21]:
# Do not hardcode a Hugging Face access token in a notebook.
# Use Colab Secrets / environment variable HF_TOKEN instead.
from huggingface_hub import login
import os

if os.getenv("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    print("HF_TOKEN is not set. Authenticate only when you need to upload results.")


HF_TOKEN is not set. Authenticate only when you need to upload results.


In [29]:
# ============================================================
# Upload to HF Bucket, with dataset-repo fallback
# ============================================================
import os, subprocess, sys
from huggingface_hub import HfApi, create_repo, upload_folder

# --- 0. Make sure we have a recent huggingface_hub ---
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub>=0.30"],
    check=False,
)

# Re-import after upgrade
import importlib
import huggingface_hub
importlib.reload(huggingface_hub)
from huggingface_hub import HfApi, create_repo, upload_folder

# --- 1. Token from Colab Secrets only ---
token = ""
if not token:
    raise RuntimeError(
        "HF_TOKEN not found. Add it in Colab Secrets (🔑 icon in left sidebar) "
        "with the name HF_TOKEN. Do not hardcode the token in a cell."
    )

# --- 2. Sanity check on OUT_DIR ---
if not os.path.isdir(OUT_DIR):
    raise RuntimeError(f"OUT_DIR '{OUT_DIR}' does not exist. Run eval cells first.")
files = sorted(os.listdir(OUT_DIR))
if not files:
    raise RuntimeError(f"OUT_DIR '{OUT_DIR}' is empty. Run eval cells first.")
print("Files to upload:")
for f in files:
    p = os.path.join(OUT_DIR, f)
    size = os.path.getsize(p) if os.path.isfile(p) else "(dir)"
    print(f"  {f:40s} {size}")

api = HfApi(token=token)
print("\nhuggingface_hub version:", huggingface_hub.__version__)

# --- 3. Try Bucket first ---
bucket_ok = False
try:
    # Some versions expose bucket creation via create_repo(repo_type="bucket")
    create_repo(
        repo_id=HF_REPO_ID,
        repo_type="bucket",
        exist_ok=True,
        token=token,
    )
    upload_folder(
        folder_path=OUT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="bucket",
        commit_message="Add blind-spot evaluation artifacts (TrashNet + Bangladesh OOD)",
        token=token,
    )
    print(f"\n✅ Uploaded to HF Bucket: https://huggingface.co/buckets/{HF_REPO_ID}")
    bucket_ok = True
except Exception as e:
    print(f"\n⚠️  Bucket upload not available on this huggingface_hub version: {e}")
    print("    Falling back to a public Dataset repo.\n")

# --- 4. Fallback to dataset repo ---
if not bucket_ok:
    create_repo(
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        exist_ok=True,
        token=token,
    )
    upload_folder(
        folder_path=OUT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        commit_message="Add blind-spot evaluation artifacts (TrashNet + Bangladesh OOD)",
        token=token,
    )
    print(f"✅ Uploaded to HF Dataset repo: https://huggingface.co/datasets/{HF_REPO_ID}")

Files to upload:
  README.md                                1870
  bangladesh_outputs.csv                   932
  cm_bangladesh_banglish.png               124816
  cm_bangladesh_en.png                     119279
  cm_trashnet_fewshot_en.png               122011
  cm_trashnet_zeroshot_en.png              121040
  metrics.json                             363
  summary.csv                              279
  trashnet_outputs.csv                     4615

huggingface_hub version: 2.1.1

⚠️  Bucket upload not available on this huggingface_hub version: Invalid repo type
    Falling back to a public Dataset repo.



No files have been modified since last commit. Skipping to prevent empty commit.


✅ Uploaded to HF Dataset repo: https://huggingface.co/datasets/rezaurrahman/bangladesh-waste-blindspot
